# AIST++ Preprocessing
### .pkl + .mp4 → frames + COCO JSON

**Влез:**
- 5 × .pkl фајлови (ground truth keypoints)
- 5 × .mp4 видеа (танцувачки клипови)

**Излез:**
- `/frames/` — извлечени слики (jpg)
- `aist_coco.json` — COCO формат анотации за евалуација
- `dataset_info.csv` — преглед на датасетот

---
**5 секвенци:**
| Видео | Жанр | Стил |
|---|---|---|
| gBR_sBM_c01_d04_mBR0_ch01 | Break | Basic Male |
| gJB_sFM_c01_d07_mJB1_ch02 | Ballet Jazz | Basic Female |
| gJS_sFM_c01_d01_mJS0_ch01 | Street Jazz | Basic Female |
| gMH_sBM_c01_d22_mMH0_ch01 | Mid Hip-hop | Basic Male |
| gLH_sFM_c01_d16_mLH0_ch01 | LA Hip-hop | Basic Female |

In [ ]:
!pip install opencv-python-headless matplotlib pandas tqdm --quiet
print('✅ Пакети инсталирани')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
 
import os
 
PKL_DIR = '/content/drive/MyDrive/AIST PKL'
MP4_DIR = '/content/drive/MyDrive/AIST MP4'
 
print('PKL фајлови:')
for f in sorted(os.listdir(PKL_DIR)):
    print(f'  {f}')
 
print('\nMP4 фајлови:')
for f in sorted(os.listdir(MP4_DIR)):
    print(f'  {f}')


In [ ]:
import os
import pickle
import numpy as np
 
SEQUENCES = [
    {
        'name': 'gBR_sBM_d04_mBR0_ch01',
        'genre': 'Break',
        'style': 'Basic Male',
        'pkl': 'gBR_sBM_cAll_d04_mBR0_ch01.pkl',
        'video': 'gBR_sBM_c01_d04_mBR0_ch01.mp4',
        'camera': 0
    },
    {
        'name': 'gJB_sFM_d07_mJB1_ch02',
        'genre': 'Ballet Jazz',
        'style': 'Basic Female',
        'pkl': 'gJB_sFM_cAll_d07_mJB1_ch02.pkl',
        'video': 'gJB_sFM_c01_d07_mJB1_ch02.mp4',
        'camera': 0
    },
    {
        'name': 'gJS_sFM_d01_mJS0_ch01',
        'genre': 'Street Jazz',
        'style': 'Basic Female',
        'pkl': 'gJS_sFM_cAll_d01_mJS0_ch01.pkl',
        'video': 'gJS_sFM_c01_d01_mJS0_ch01.mp4',
        'camera': 0
    },
    {
        'name': 'gMH_sBM_d22_mMH0_ch01',
        'genre': 'Mid Hip-hop',
        'style': 'Basic Male',
        'pkl': 'gMH_sBM_cAll_d22_mMH0_ch01.pkl',
        'video': 'gMH_sBM_c01_d22_mMH0_ch01.mp4',
        'camera': 0
    },
    {
        'name': 'gLH_sFM_d16_mLH0_ch01',
        'genre': 'LA Hip-hop',
        'style': 'Basic Female',
        'pkl': 'gLH_sFM_cAll_d16_mLH0_ch01.pkl',
        'video': 'gLH_sFM_c01_d16_mLH0_ch01.mp4',
        'camera': 0
    }
]
 
print('=' * 55)
print('ПРОВЕРКА НА ФАЈЛОВИ:')
print('=' * 55)
 
all_ok = True
for seq in SEQUENCES:
    pkl_ok = os.path.exists(seq['pkl'])
    mp4_ok = os.path.exists(seq['video'])
    status = '✅' if pkl_ok and mp4_ok else '❌'
    print(f"{status} {seq['genre']:15s} | pkl={'✅' if pkl_ok else '❌'} | mp4={'✅' if mp4_ok else '❌'}")
    if not pkl_ok or not mp4_ok:
        all_ok = False
 
print('=' * 55)
if all_ok:
    print('✅ Сите фајлови се присутни — може да продолжиме!')
else:
    print('❌ Недостасуваат фајлови — провери го качувањето')


In [ ]:
with open(SEQUENCES[0]['pkl'], 'rb') as f:
    sample = pickle.load(f)
 
print('Клучеви во .pkl фајлот:')
for key, val in sample.items():
    if hasattr(val, 'shape'):
        print(f'  {key}: shape={val.shape}, dtype={val.dtype}')
    else:
        print(f'  {key}: {type(val)}')
 
print(f'\nkeypoints2d shape: {sample["keypoints2d"].shape}')
print('  (камери, frames, keypoints, x/y/conf)')
print(f'\nВкупно frames: {sample["keypoints2d"].shape[1]}')
print(f'Траење при 60fps: {sample["keypoints2d"].shape[1]/60:.1f} секунди')


In [ ]:
kpts = sample['keypoints2d'][0][0]
 
keypoint_names = [
    'нос', 'лево_око', 'десно_око', 'лево_уво', 'десно_уво',
    'лево_рамо', 'десно_рамо', 'лев_лакт', 'десен_лакт',
    'лев_зглоб', 'десен_зглоб', 'лев_колк', 'десен_колк',
    'лево_колено', 'десно_колено', 'лев_глуждот', 'десен_глуждот'
]
 
print('Keypoints на Frame 0, Камера 0 (Break, Basic Male):')
print(f'{"Keypoint":<15} {"x":>8} {"y":>8} {"conf":>8}')
print('-' * 42)
for name, kpt in zip(keypoint_names, kpts):
    print(f'{name:<15} {kpt[0]:>8.1f} {kpt[1]:>8.1f} {kpt[2]:>8.3f}')


In [ ]:
import cv2
from tqdm import tqdm
 
FRAME_STEP = 6
MAX_FRAMES = 100
 
os.makedirs('frames', exist_ok=True)
 
frame_info = {}
 
for seq in SEQUENCES:
    seq_name = seq['name']
    genre = seq['genre']
     
    print(f'\nОбработка: {genre}...')
     
    cap = cv2.VideoCapture(seq['video'])
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
     
    print(f'  FPS: {fps}, Вкупно frames: {total}, Резолуција: {w}x{h}')
     
    saved_frames = []
    frame_idx = 0
    saved_count = 0
     
    while cap.isOpened() and saved_count < MAX_FRAMES:
        ret, frame = cap.read()
        if not ret:
            break
         
        if frame_idx % FRAME_STEP == 0:
            
            img_name = f'{seq_name}_frame_{saved_count:04d}.jpg'
            img_path = f'frames/{img_name}'
            cv2.imwrite(img_path, frame)
             
            saved_frames.append({
                'frame_idx': frame_idx,
                'saved_idx': saved_count,
                'img_name': img_name,
                'img_path': img_path,
                'width': w,
                'height': h
            })
            saved_count += 1
         
        frame_idx += 1
     
    cap.release()
    frame_info[seq_name] = saved_frames
    print(f'  ✅ Зачувани: {saved_count} frames')
 
total_frames = sum(len(v) for v in frame_info.values())
print(f'\n✅ Вкупно извлечени frames: {total_frames}')


In [ ]:
 
matched_data = {}
 
for seq in SEQUENCES:
    seq_name = seq['name']
    cam_idx = seq['camera']
     
    
    with open(seq['pkl'], 'rb') as f:
        data = pickle.load(f)
     
    keypoints2d = data['keypoints2d']
    cam_kpts = keypoints2d[cam_idx]
    total_kpt_frames = cam_kpts.shape[0]
     
    matched = []
    for finfo in frame_info[seq_name]:
        orig_idx = finfo['frame_idx']
         
        
        if orig_idx < total_kpt_frames:
            kpts = cam_kpts[orig_idx]
            matched.append((finfo, kpts))
     
    matched_data[seq_name] = matched
    print(f'✅ {seq["genre"]:15s}: {len(matched)} frame-keypoint парови')
 
print(f'\nВкупно парови: {sum(len(v) for v in matched_data.values())}')


In [ ]:
import json
 
KEYPOINT_NAMES = [
    'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
    'left_knee', 'right_knee', 'left_ankle', 'right_ankle'
]
 
SKELETON = [
    [16,14],[14,12],[17,15],[15,13],[12,13],
    [6,12],[7,13],[6,7],[6,8],[7,9],
    [8,10],[9,11],[2,3],[1,2],[1,3],[2,4],[3,5],[4,6],[5,7]
]
 
coco_images = []
coco_annotations = []
img_id = 0
ann_id = 0
 
sequence_mapping = []
 
CONF_THRESHOLD = 0.3
 
for seq in SEQUENCES:
    seq_name = seq['name']
    genre = seq['genre']
     
    for finfo, kpts in matched_data[seq_name]:
        
        coco_images.append({
            'id': img_id,
            'file_name': finfo['img_name'],
            'width': finfo['width'],
            'height': finfo['height'],
            'genre': genre,
            'style': seq['style'],
            'sequence': seq_name
        })
         
        
        kpts_coco = []
        num_visible = 0
        xs, ys = [], []
         
        for kpt in kpts:
            x, y, conf = float(kpt[0]), float(kpt[1]), float(kpt[2])
            v = 2 if conf > CONF_THRESHOLD else 1
            kpts_coco.extend([x, y, v])
            if v == 2:
                num_visible += 1
                xs.append(x)
                ys.append(y)
         
        
        if xs and ys:
            x_min, x_max = min(xs), max(xs)
            y_min, y_max = min(ys), max(ys)
            padding = 20
            bbox = [
                max(0, x_min - padding),
                max(0, y_min - padding),
                min(finfo['width'], x_max - x_min + 2*padding),
                min(finfo['height'], y_max - y_min + 2*padding)
            ]
            area = bbox[2] * bbox[3]
        else:
            bbox = [0, 0, finfo['width'], finfo['height']]
            area = finfo['width'] * finfo['height']
         
        
        coco_annotations.append({
            'id': ann_id,
            'image_id': img_id,
            'category_id': 1,
            'keypoints': kpts_coco,
            'num_keypoints': num_visible,
            'bbox': [float(b) for b in bbox],
            'area': float(area),
            'iscrowd': 0
        })
         
        
        sequence_mapping.append({
            'img_id': img_id,
            'sequence': seq_name,
            'genre': genre,
            'frame_idx': finfo['frame_idx']
        })
         
        img_id += 1
        ann_id += 1
 
coco_json = {
    'info': {
        'description': 'AIST++ subset — 5 dance sequences',
        'version': '1.0',
        'year': 2026
    },
    'images': coco_images,
    'annotations': coco_annotations,
    'categories': [{
        'id': 1,
        'name': 'person',
        'supercategory': 'person',
        'keypoints': KEYPOINT_NAMES,
        'skeleton': SKELETON
    }]
}
 
with open('aist_coco.json', 'w') as f:
    json.dump(coco_json, f, indent=2)
 
with open('sequence_mapping.json', 'w') as f:
    json.dump(sequence_mapping, f, indent=2)
 
print('✅ COCO JSON зачуван: aist_coco.json')
print(f'   Слики:      {len(coco_images)}')
print(f'   Анотации:   {len(coco_annotations)}')
print()
print('✅ Sequence mapping зачуван: sequence_mapping.json')
print('   (ќе го користиш за Action Recognition во Недела 3)')


In [ ]:
import pandas as pd
 
rows = []
for seq in SEQUENCES:
    seq_name = seq['name']
    n_frames = len(matched_data[seq_name])
     
    
    all_conf = []
    for _, kpts in matched_data[seq_name]:
        all_conf.extend(kpts[:, 2].tolist())
    avg_conf = np.mean(all_conf)
     
    rows.append({
        'Жанр': seq['genre'],
        'Стил': seq['style'],
        'Frames': n_frames,
        'Просечен conf': f'{avg_conf:.3f}',
        '.pkl': seq['pkl'],
        '.mp4': seq['video']
    })
 
df = pd.DataFrame(rows)
print('=== ПРЕГЛЕД НА ДАТАСЕТОТ ===')
print(df.to_string(index=False))
 
df.to_csv('dataset_info.csv', index=False)
print('\n✅ dataset_info.csv зачуван')


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
 
SKELETON_PAIRS = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(6,8),(7,9),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(12,14),(13,15),(14,16)
]
 
fig, axes = plt.subplots(1, 5, figsize=(20, 6))
 
for ax, seq in zip(axes, SEQUENCES):
    seq_name = seq['name']
     
    if not matched_data[seq_name]:
        ax.set_title(f"{seq['genre']}\nНема frames")
        continue
     
    
    frame_n = min(10, len(matched_data[seq_name])-1)
    finfo, kpts = matched_data[seq_name][frame_n]
     
    
    img = cv2.imread(finfo['img_path'])
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
     
    
    for i, j in SKELETON_PAIRS:
        x1, y1, c1 = kpts[i]
        x2, y2, c2 = kpts[j]
        if c1 > 0.3 and c2 > 0.3:
            ax.plot([x1, x2], [y1, y2], 'cyan', linewidth=2, alpha=0.8)
     
    
    for kpt in kpts:
        x, y, conf = kpt
        if conf > 0.3:
            ax.scatter(x, y, c='red', s=30, zorder=5)
     
    ax.set_title(f"{seq['genre']}\n{seq['style']}", fontsize=10)
    ax.axis('off')
 
plt.suptitle('AIST++ Ground Truth Keypoints — по жанр', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('aist_preview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Визуализација зачувана: aist_preview.png')


In [ ]:
import shutil
 
OUTPUT_DIR = '/content/drive/MyDrive/AIST Output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
shutil.copy('aist_coco.json', f'{OUTPUT_DIR}/aist_coco.json')
shutil.copy('sequence_mapping.json', f'{OUTPUT_DIR}/sequence_mapping.json')
shutil.copy('dataset_info.csv', f'{OUTPUT_DIR}/dataset_info.csv')
shutil.copy('aist_preview.png', f'{OUTPUT_DIR}/aist_preview.png')
 
frames_output = f'{OUTPUT_DIR}/frames'
if os.path.exists(frames_output):
    shutil.rmtree(frames_output)
shutil.copytree('frames', frames_output)
 
print('✅ Сè зачувано на Drive во: AIST Output/')
print()
for f in os.listdir(OUTPUT_DIR):
    print(f'  {f}')
print()
print('Следно — Notebook 2: Pose Estimation евалуација')
print('  Влез: AIST Output/frames + AIST Output/aist_coco.json')
